# Week 20 · Notebook 02: Bedrock Agents & Guardrails

# Requirements: pip install boto3 numpy pandas

# ⚠️ REQUIRES: AWS

> 💰 COST WARNING: set an AWS budget alert before running, agents bill per token and provisioned resources meter while idle; delete what you create.


## What you build

Create a Bedrock **Guardrail** (denied topics, content filters) and apply it to test prompts (insults, off-topic, on-topic), counting blocked vs allowed. Then create a Bedrock **Agent** via boto3, with cleanup cells so nothing meters while idle. Dry-run uses a deterministic local guardrail so the counts are reproducible without AWS.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1])) # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import numpy as np
import pandas as pd
import boto3
from zoro import data

SEED = 20
rng = np.random.default_rng(SEED)
REGION = os.environ.get("AWS_REGION", "us-east-1")

def probe_aws():
    try:
        boto3.client("bedrock", region_name=REGION).list_foundation_models()
        return True
    except Exception as e: # noqa: BLE001
        print("AWS probe failed:", type(e).__name__, e)
        return False

BEDROCK_READY = probe_aws()
if not BEDROCK_READY:
    print("⚠️ AWS credentials not found, running in DRY-RUN mode.")
    print("Run `aws configure` (or `aws sso login`), then restart the kernel.")
    print("Also enable models under Bedrock → Model access (AccessDeniedException = forgot this).")

runtime = boto3.client("bedrock-runtime", region_name=REGION)
print("boto3 ready; region =", REGION)


## Create a Guardrail

Guardrails are configurable safety filters applied to model I/O: denied topics, content filters (insults/hate/violence), PII redaction, and custom word filters. We block off-topic queries and insults.


In [ ]:
bedrock = boto3.client("bedrock", region_name=REGION)

guardrail_id, guardrail_version = None, None
if BEDROCK_READY:
    try:
        resp = bedrock.create_guardrail(
            name="zoro-support-guardrail",
            description="Blocks insults and off-topic queries for the ZoroLogistics support agent.",
            topicPolicyConfig={
                "topicsConfig": [
                    {"name": "OffTopic",
                     "definition": "Queries unrelated to freight, shipping, or ZoroLogistics support.",
                     "examples": ["What is the weather in Paris?", "Write me a poem about cats."],
                     "type": "DENY"},
                ]
            },
            contentPolicyConfig={
                "filtersConfig": [
                    {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                    {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                    {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                ]
            },
            wordPolicyConfig={
                "wordsConfig": [{"text": "competitorpricing"}],
                "managedWordListsConfig": [{"type": "PROFANITY"}],
            },
        )
        guardrail_id = resp["guardrailId"]
        guardrail_version = resp["version"]
        print("✅ Created guardrail:", guardrail_id, "v" + str(guardrail_version))
    except Exception as e:  # noqa: BLE001
        print("⚠️ create_guardrail failed:", type(e).__name__, e)
else:
    print("DRY-RUN: skipped create_guardrail (no AWS credentials).")


## Test blocked prompts

Apply the guardrail to insult, off-topic, and legitimate prompts, and count how many are blocked vs allowed. This is the AWS answer to "how do I keep the agent from saying the wrong thing."


In [ ]:
def apply_guardrail(guardrail_id, version, prompt):
    resp = runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=version,
        source="INPUT",
        content=[{"text": {"text": prompt}}],
    )
    return resp["action"]  # "GUARDRAIL_INTERVENED" or "NONE"

def local_guardrail(prompt):
    q = prompt.lower()
    insult_words = ["idiot", "garbage", "stupid", "useless"]
    if any(w in q for w in insult_words):
        return "GUARDRAIL_INTERVENED"
    freight_kw = ["shipment", "refund", "track", "freight", "delivery", "policy", "pallet", "bill"]
    if not any(w in q for w in freight_kw):
        return "GUARDRAIL_INTERVENED"
    return "NONE"

blocked_prompts = [
    "You are a useless idiot and your company is garbage.",
    "Write me a recipe for chocolate cake.",
    "Tell me a joke about pirates.",
]
allowed_prompts = [
    "Where is my shipment S0000001?",
    "What is the refund policy for a late delivery?",
]

blocked = allowed = 0
for p in blocked_prompts:
    action = apply_guardrail(guardrail_id, guardrail_version, p) if (BEDROCK_READY and guardrail_id) else local_guardrail(p)
    blocked += int(action == "GUARDRAIL_INTERVENED")
    print(f"  [{action}] {p[:50]}")
for p in allowed_prompts:
    action = apply_guardrail(guardrail_id, guardrail_version, p) if (BEDROCK_READY and guardrail_id) else local_guardrail(p)
    allowed += int(action == "NONE")
    print(f"  [{action}] {p[:50]}")


## Create a Bedrock Agent (optional)

A Bedrock Agent = model + instructions + knowledge base + action groups (Lambda). Creation needs an execution role ARN (`BEDROCK_AGENT_ROLE_ARN`). The cleanup cell at the end deletes it.


In [ ]:
agent_id = None
if BEDROCK_READY:
    role_arn = os.environ.get("BEDROCK_AGENT_ROLE_ARN", "")
    if role_arn:
        try:
            agent_client = boto3.client("bedrock-agent", region_name=REGION)
            resp = agent_client.create_agent(
                agentName="zoro-support-agent",
                foundationModel="anthropic.claude-3-5-sonnet-20241022-v2:0",
                instruction=(
                    "You are the ZoroLogistics support agent. Track shipments, answer "
                    "policy questions from the knowledge base, and never invent a tracking number."
                ),
                agentResourceRoleArn=role_arn,
            )
            agent_id = resp["agent"]["agentId"]
            print("✅ Created agent:", agent_id)
        except Exception as e:  # noqa: BLE001
            print("⚠️ create_agent failed:", type(e).__name__, e)
    else:
        print("SKIPPED create_agent (set BEDROCK_AGENT_ROLE_ARN).")
else:
    print("DRY-RUN: skipped create_agent (no AWS credentials).")


## Cleanup: delete what you created

Endpoints, provisioned capacity, and agents meter even when idle. Always tear down. These cells are safe to re-run (they no-op if nothing was created).


In [ ]:
if agent_id:
    try:
        boto3.client("bedrock-agent", region_name=REGION).delete_agent(agentId=agent_id)
        print("✅ Deleted agent", agent_id)
    except Exception as e:  # noqa: BLE001
        print("⚠️ delete_agent failed:", type(e).__name__, e)

if guardrail_id:
    try:
        bedrock.delete_guardrail(guardrailIdentifier=guardrail_id, guardrailVersion=guardrail_version)
        print("✅ Deleted guardrail", guardrail_id)
    except Exception as e:  # noqa: BLE001
        print("⚠️ delete_guardrail failed:", type(e).__name__, e)


In [ ]:
# Final numbers: blocked vs allowed prompt counts.
print(f"BLOCKED: {blocked}  ALLOWED: {allowed}")
